# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [14]:
import sys
import pandas as pd
from IPython.display import display

from routing.config import ORIGIN, ORIGINS, DESTINATION
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph, save_summary
from routing.traffic_quality import plan_corridor_probes, audit_traffic_snapshot

print("Kernel:", sys.executable)


Kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python


## ۱. بارگذاری گراف OSM (بدون TomTom)


In [15]:
RADIUS_M = 2000

G = load_road_graph(radius_m=RADIUS_M)

print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

assert G.is_directed()
assert G.is_multigraph()

print("PASS: OSM Graph")

Graph: MultiDiGraph
Nodes: 2569
Edges: 5850
PASS: OSM Graph


## ۲. برنامه‌ریزی Probeها بدون API


In [16]:
REQUEST_LIMIT = 12

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
)

print("Planned requests:", len(plan))

display(pd.DataFrame(plan))

Planned requests: 12


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(109869, 265563774, 0)",51.495550,-0.181906,134.42,residential,Gloucester Road,1.3
1,"(26389375, 34519894, 0)",51.498980,-0.174206,181.39,unclassified,Royal Albert Hall,2.2
2,"(1360237602, 3925964624, 0)",51.495273,-0.174187,68.82,residential,South Kensington,0.9
3,"(1027237558, 1116949690, 0)",51.496115,-0.180221,187.61,residential,Gloucester Road,18.2
4,"(1691189781, 20821126, 0)",51.500769,-0.176459,72.93,residential,Royal Albert Hall,45.6
5,"(34519894, 6280192964, 0)",51.496979,-0.173807,266.83,unclassified,South Kensington,27.3
6,"(1356901661, 109869, 0)",51.495093,-0.182533,41.83,trunk,Gloucester Road,23.2
7,"(20947178, 29353841, 0)",51.500597,-0.178215,200.15,residential,Royal Albert Hall,49.2
8,"(26389442, 34519894, 0)",51.498196,-0.173748,41.05,residential,South Kensington,31.4
9,"(1027237558, 860183693, 0)",51.496988,-0.179934,67.30,residential,Gloucester Road,49.3


## ۳. اجرای کنترل‌شده جمع‌آوری


In [17]:
RUN_LIVE_TRAFFIC = False

selected_edges = [
    row["edge_id"]
    for row in plan
]

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G,
            client,
            max_requests=REQUEST_LIMIT,
            selected_edge_ids=selected_edges,
        )

    save_summary(summary)

else:
    snapshot, summary = collect_traffic_snapshot(
        G,
        None,
        max_requests=0,
        selected_edge_ids=selected_edges,
    )

print("Requests:", summary["requests_made"])
print("Counts:", summary["counts"])

Requests: 0
Counts: {'missing': 5850, 'red': 0, 'yellow': 0, 'clear': 0, 'closed': 0}


## ۴. Audit کیفیت و قابلیت آماده‌سازی وزن‌ها



In [18]:
report = audit_traffic_snapshot(
    G,
    snapshot,
    summary
)

print(
    "Graph identity:",
    report["graph_identity_ok"]
)

print(
    "Costs valid:",
    report["costs_ok"]
)

print(
    "Algorithm smoke test ready:",
    report["algorithm_smoke_test_ready"]
)

print(
    "Final traffic comparison ready:",
    report["final_traffic_comparison_ready"]
)

print("Warnings:", report["warnings"])

Graph identity: True
Costs valid: True
Algorithm smoke test ready: True
Final traffic comparison ready: False
Warnings: ['Observed traffic coverage is sparse; most travel times are not measured', 'TomTom geometry point order is not authoritative legal direction proof']


## ۵. نقشه تعاملی و لایه‌های ترافیک


In [19]:
traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot
)

display(traffic_widget)

In [20]:
print("Origin:", ui_state["origin_node"])
print("Destination:", ui_state["destination_node"])
print("Last click:", ui_state["last_click_latlon"])

if ui_state["last_click_latlon"] is not None:
    assert ui_state["origin_node"] in G
    assert ui_state["destination_node"] in G
    print("PASS: Interactive Map")
else:
    print("No double-click received yet")

Origin: 25291778
Destination: 26389375
Last click: None
No double-click received yet


## ۶. سه سناریو، دو Objective، بدون اجرای الگوریتم‌ها



In [21]:
free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow"
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured"
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured"
)

print(
    "Free flow:",
    free_flow_time.number_of_edges()
)

print(
    "Measured time:",
    measured_time.number_of_edges()
)

print(
    "Measured distance:",
    measured_distance.number_of_edges()
)

print("PASS: Routing graphs prepared") 

Free flow: 5850
Measured time: 5850
Measured distance: 5850
PASS: Routing graphs prepared
